In [ ]:
import pandas as pd
import numpy as np

from snp_analysis_tools_sherlock import *

import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
fname = '~/git/coalescence-pilot-mgx/midas2_output/mergev2/species/species_relative_abundance.tsv'
df_abundance = pd.read_csv(fname, delimiter = '\t')
df_abundance
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

df_abundance_melted = pd.melt(df_abundance, 
                                  var_name = 'sample', 
                                  id_vars='species_id',
                                  value_name = 'relative_abundance')


df_abundance_melted = transform_df(df_abundance_melted)
#df_abundance_melted['speciesid']# -sample'] = df_abundance_melted['species_id'] + '-' + df_abundance_melted['sample']
df_abundance_melted.head()

In [ ]:
fnames = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/dir_selectionv2/*')
dfs = []
for fname in fnames:
    df = pd.read_csv(fname) #.set_index(sample)
    species = fname.split('/')[-1].split('_shifts.csv')[0]
    df_abundance_sp = df_abundance_melted.loc[df_abundance_melted['species_id'] == int(species),:].set_index('sample')

    df['species_id'] = species
    df['species_abundance']= df['sample'].transform(lambda x: df_abundance_sp.loc[x, 'relative_abundance'])
    df['Lineage'] = df_abundance_sp['Lineage'].values[0]
    df['species'] = df_abundance_sp['species'].values[0]
    dfs.append(df)

full_df = pd.concat(dfs)
full_df['minor_strain_freq'] = full_df['strain_freq']
full_df.loc[full_df['minor_strain_freq']>.5,'minor_strain_freq'] = 1. - full_df.loc[full_df['minor_strain_freq']>.5,'minor_strain_freq'] 


In [ ]:

full_df = full_df.loc[full_df['passage'] > 3,:]

In [ ]:
len(full_df['species_id'].unique())

In [ ]:

full_dfgood = full_df.loc[full_df['total_shift'] < .08,:]
full_dfgood = full_dfgood.loc[full_dfgood['passage'] == 7.,:]
p = iqplot.ecdf(full_dfgood,q = 'minor_strain_freq',)
bokeh.io.show(p)

In [ ]:
full_dfgood 

In [ ]:

full_dfgood['counts'] = 1
full_dfgood['parent_subjects'] = full_dfgood['inoculumn'].transform(lambda x: '-'.join(x.split('-')[:-1]))
full_dfgood['minor_strain_alive'] = full_dfgood['minor_strain_freq']>0
full_dfgood['minor_strain_dead']  = full_dfgood['minor_strain_freq']==0
full_dfgoodgr = full_dfgood.groupby(['species_id', 'type_meso', 'media', 'parent_media',
                                     'parent_subjects', 'inoculumn']).sum().reset_index()
#full_dfgoodgr_alive = full_dfgoodgr.loc[full_dfgoodgr['minor_strain_alive']>0,:]
full_dfgoodgr_alive = full_dfgoodgr.loc[full_dfgoodgr['counts']>1,:]
full_dfgoodgr_alive['avg_freq'] = full_dfgoodgr_alive['minor_strain_freq']/full_dfgoodgr_alive['counts']
full_dfgoodgr_alive['avg_alive'] = full_dfgoodgr_alive['minor_strain_alive']/full_dfgoodgr_alive['counts']

#full_dfgoodgr_alive = full_dfgoodgr_alive.loc[full_dfgoodgr_alive['avg_alive'] == 1,:]
#full_dfgoodgr_alive.sort_values(by='species_id')

In [ ]:
full_dfgoodgr_dead = full_dfgoodgr.loc[full_dfgoodgr['counts']>1,:]

full_dfgoodgr_dead['avg_freq'] = full_dfgoodgr_dead['minor_strain_freq']/full_dfgoodgr_dead['counts']
full_dfgoodgr_dead['avg_dead'] = full_dfgoodgr_dead['minor_strain_dead']/full_dfgoodgr_dead['counts']
len(full_dfgoodgr_dead.loc[full_dfgoodgr_dead['avg_freq']==0.,:])
full_dfgoodgr_dead.loc[full_dfgoodgr_dead['avg_dead']==1.,:].to_csv('dead.csv')

In [ ]:
len(full_dfgoodgr_dead.loc[full_dfgoodgr_dead['avg_dead'] == 1,:])/len(full_dfgoodgr_dead)

In [ ]:
len(full_dfgoodgr_dead.loc[full_dfgoodgr_dead['avg_dead'] == 1,'species_id'].unique())

In [ ]:
full_dfgoodgr_alive.loc[full_dfgoodgr_alive['avg_alive'] == 1,:].to_csv('alive.csv')

In [ ]:
1-.48-.23

In [ ]:
len(full_dfgoodgr_alive.loc[full_dfgoodgr_alive['avg_alive'] < 1,'species_id'].unique())

In [ ]:
len(full_dfgoodgr_alive.loc[full_dfgoodgr_alive['avg_alive'] == 1,'species_id'].unique())

In [ ]:
np.intersect1d(full_dfgoodgr_alive.loc[full_dfgoodgr_alive['avg_alive'] == 1,'species_id'].unique(), 
               full_dfgoodgr_alive.loc[full_dfgoodgr_alive['avg_alive'] < 1,'species_id'].unique())

In [ ]:
full_dfgoodgr_alive

In [ ]:
p = iqplot.ecdf(full_dfgoodgr_alive, q = 'avg_freq' )#, bins = 20)
bokeh.io.show(p)

In [ ]:
len(full_dfgoodgr_alive['species_id'].unique())

In [ ]:
b = full_dfgood.loc[full_dfgood['type_meso'] == 'AE-AF-mGAM-mGAM',:]
b = b.loc[b['passage'] == 7,:]
b = b.loc[b['species_id'] == '100099',:]
b

In [ ]:
two_rep = full_dfgoodgr2.loc[full_dfgoodgr2['counts'] >= 2,:].reset_index()
two_rep['match'] = np.abs(two_rep['overall_shift']) == two_rep['counts']

len(two_rep.loc[two_rep['match'],:])

In [ ]:
two_rep['species_id'].unique()

In [ ]:
df_metadata['species_id'].unique()

In [ ]:
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata['species_id'] = df_metadata['species_id'].astype(str)

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

two_rep_tr = transform_df(two_rep)
#two_rep['Lineage'] = two_rep['species_id'].transform(lambda x: df_metadata.loc[int(df_metadata['species_id'] == x),'Lineage'].values[0])


In [ ]:
two_rep['counts_again'] = 1 
two_rep_by_sp_media = two_rep.groupby([ 'species','family','phyla']).sum()

two_rep_by_sp_media['fraction_sel'] =  two_rep_by_sp_media['match']/two_rep_by_sp_media['counts_again']

In [ ]:
two_rep_by_sp_media = two_rep_by_sp_media.loc[two_rep_by_sp_media['counts_again'] > 5,:]

In [ ]:
p = iqplot.strip(data = two_rep_by_sp_media.reset_index(), cats = ['species',],q = 'fraction_sel',
                 color_column = 'family',
                 width = 700,
                 marker_kwargs=dict(line_color='black',size=10),
    )

p.x_range = bokeh.models.Range1d(0,1)
bokeh.io.show(p)

In [ ]:
b = two_rep[['species_id', 'type_meso', 'media', 'parent_media', 'parent_subjects', 'passage', 'overall_shift']].loc[two_rep['species_id'] == '101346',:]
c = b.loc[b['media'] == 'mGAM']
d = c.loc[c['parent_media'] == 'mGAM']
d

In [ ]:
two_rep_gr2.index.values

In [ ]:
two_rep['counts3'] = 1
two_rep_gr2 = two_rep.reset_index().groupby(['species_id', 'media', 'parent_media']).sum()
two_rep_gr2 = two_rep_gr2.sort_values(by='counts3', ascending = False)
two_rep_gr2= two_rep_gr2.loc[two_rep_gr2['counts3'] > 2,:]
two_rep_gr2[['counts3']]

In [ ]:
one_rep = full_dfgoodgr2.loc[full_dfgoodgr2['counts'] <2,:]
one_rep

In [ ]:
len(two_rep.reset_index()['species_id'].unique())

In [ ]:
two_rep_media 

In [ ]:
two_rep_media = two_rep.groupby(['species_id', 'parent_media', 'inoculumn', ]).sum()
two_rep_media = two_rep_media.loc[two_rep_media['counts'] >=4,:]

In [ ]:
len(two_rep_media)

In [ ]:
two_rep_media['match'] = two_rep_media['counts'] ==  two_rep_media['overall_shift'].abs()
len(two_rep_media.loc[two_rep_media['match'],:])

In [ ]:
two_rep_media = two_rep_media.loc[two_rep_media['overall_shift'].abs() != 2,:]

In [ ]:
len(two_rep_media) 

In [ ]:
two_rep_media

In [ ]:
two_rep_media.loc[two_rep_media['overall_shift'] == 0,:]

In [ ]:
two_rep_media_acr = two_rep.groupby(['species_id', 'media', 'parent_subjects']).sum()
two_rep_media_acr = two_rep_media_acr.loc[two_rep_media_acr['counts'] ==4,:]
two_rep_media_acr